In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# load the data
df = pd.read_csv("../data/raw/11_Baseketball_Team_Performance_Analysis.csv")
df.head()

In [0]:
# check for missing values
print(df.isnull().sum())

# fill missing values with median
df = df.fillna(df.median(numeric_only=True))
print("\nNulls after filling:", df.isnull().sum().sum())

In [0]:
# check duplicates
print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [0]:
# visualize outliers for key metrics
cols_to_check = ['offensive_rating', 'defensive_rating', 'net_rating', 'points', 
                 'opponent_points', 'turnovers', 'total_rebounds', 'assists', 'steals', 'blocks']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(cols_to_check):
    sns.boxplot(x=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

In [0]:
# remove outliers using IQR method
for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

In [0]:
# check after outlier treatment
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(cols_to_check):
    sns.boxplot(x=df[col], ax=axes[i], color='lightgreen')
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

In [0]:
# create some new features with safe division
df['point_diff'] = df['points'] - df['opponent_points']
df['assist_turnover_ratio'] = np.where(df['turnovers'] == 0, 0, df['assists'] / df['turnovers'])

df['home_win_pct'] = np.where((df['home_wins'] + df['home_losses']) == 0, 0.5, df['home_wins'] / (df['home_wins'] + df['home_losses']))
df['away_win_pct'] = np.where((df['away_wins'] + df['away_losses']) == 0, 0.5, df['away_wins'] / (df['away_wins'] + df['away_losses']))
df['home_advantage'] = df['home_win_pct'] - df['away_win_pct']

df['conf_win_pct'] = np.where((df['conference_wins'] + df['conference_losses']) == 0, 0.5, df['conference_wins'] / (df['conference_wins'] + df['conference_losses']))
df['scoring_efficiency'] = np.where(df['field_goal_attempts'] == 0, 0, df['points'] / df['field_goal_attempts'])
df['defensive_pressure'] = df['steals'] + df['blocks']

In [0]:
# save cleaned data
df.to_csv("../data/processed/cleaned_basketball_data.csv", index=False)